In [1]:
# create_dataset.py
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import warnings 
warnings.filterwarnings("ignore")
import joblib

In [2]:
n = 500  # number of samples

names = [f"Patient_{i}" for i in range(n)]
dob = [datetime(1960,1,1) + timedelta(days=random.randint(0, 18250)) for _ in range(n)]
email = [f"user{i}@example.com" for i in range(n)]
glucose = np.random.randint(70, 200, n)
haemoglobin = np.random.uniform(10, 17, n)
cholesterol = np.random.randint(150, 300, n)

In [3]:
remarks = []
for g, h, c in zip(glucose, haemoglobin, cholesterol):
    if g > 140:
        remarks.append("Diabetes Risk")
    elif h < 12:
        remarks.append("Anaemia Risk")
    elif c > 240:
        remarks.append("Heart Risk")
    else:
        remarks.append("Healthy")

df = pd.DataFrame({
    "name": names,
    "dob": dob,
    "email": email,
    "glucose": glucose,
    "haemoglobin": haemoglobin,
    "cholesterol": cholesterol,
    "remarks": remarks
})

df.to_csv("health_data.csv", index=False)
print("Dataset created and saved as health_data.csv")

Dataset created and saved as health_data.csv


In [4]:
# Load dataset
df = pd.read_csv("health_data.csv")

# Features and target
X = df[['glucose', 'haemoglobin', 'cholesterol']]
y = df['remarks'].astype('category').cat.codes

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define models
models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}


In [5]:
best_model = None
best_acc = 0

# Train and evaluate each model
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)

    print(f"{name} - Train Accuracy: {train_acc:.2f}, Test Accuracy: {test_acc:.2f}")
    print(classification_report(y_test, y_pred_test))

    # Select best model based on test accuracy
    if test_acc > best_acc:
        best_acc = test_acc
        best_model = model
        best_name = name

# Save best model
joblib.dump(best_model, "health_model.pkl")
print(f"Best model: {best_name} with Test Accuracy: {best_acc:.2f}")
print("Model saved as health_model.pkl")

RandomForest - Train Accuracy: 1.00, Test Accuracy: 0.99
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        21
           1       1.00      1.00      1.00        43
           2       0.96      1.00      0.98        22
           3       1.00      0.93      0.96        14

    accuracy                           0.99       100
   macro avg       0.99      0.98      0.99       100
weighted avg       0.99      0.99      0.99       100

LogisticRegression - Train Accuracy: 0.97, Test Accuracy: 0.96
              precision    recall  f1-score   support

           0       0.91      1.00      0.95        21
           1       1.00      0.95      0.98        43
           2       0.95      0.91      0.93        22
           3       0.93      1.00      0.97        14

    accuracy                           0.96       100
   macro avg       0.95      0.97      0.96       100
weighted avg       0.96      0.96      0.96       100

SVM - Trai